# Error Propagation Module Demonstrations

This notebook showcases the full capabilities of the `lab_math_tools.error_propagation` module:
- **`propagate_uncertainty`**: Propagates measurement uncertainty through any scalar-valued function using numerical partial derivatives. Supports both statistical (RSS) and absolute (worst-case) combination rules.
- **`relative_uncertainty`**: Computes the dimensionless fractional uncertainty $|\Delta f / f(\mathbf{x})|$.

### Key formulas

**Statistical (Root-Sum-Square, RSS):**
$$\Delta f = \sqrt{\sum_i \left(\frac{\partial f}{\partial x_i} \cdot \Delta x_i\right)^2}$$

**Absolute (worst-case):**
$$\Delta f = \sum_i \left|\frac{\partial f}{\partial x_i} \cdot \Delta x_i\right|$$

All partial derivatives are computed numerically using the SAAP central-difference method from the `derivatives` module.


In [1]:
# Imports
import numpy as np
from lab_math_tools.error_propagation import propagate_uncertainty, relative_uncertainty


---
## 1. `propagate_uncertainty` — Scalar Case ($\mathbb{R} \to \mathbb{R}$)

For a single-variable function $f(x)$, the uncertainty formula reduces to:
$$\Delta f = |f'(x)| \cdot \Delta x$$

The function detects scalar inputs automatically — no array wrapping needed.

### 1.1 Power law: $f(x) = x^2$
Exact sensitivity: $|f'(x)| = 2|x|$ so at $x = 3.0 \pm 0.1$: $\Delta f = 2 \cdot 3 \cdot 0.1 = 0.6$.


In [2]:
f  = lambda x: x**2
x  = 3.0
dx = 0.1

delta_f = propagate_uncertainty(f, x, dx)
exact   = abs(2 * x) * dx

print(f"f(x) = x^2   at   x = {x} +/- {dx}")
print(f"  Propagated uncertainty  : {delta_f:.6f}")
print(f"  Exact                   : {exact:.6f}")
print(f"  Error                   : {abs(delta_f - exact):.2e}")


f(x) = x^2   at   x = 3.0 +/- 0.1
  Propagated uncertainty  : 0.600000
  Exact                   : 0.600000
  Error                   : 3.93e-12


### 1.2 Exponential: $f(x) = e^x$
Exact sensitivity: $|f'(x)| = e^x$ so at $x = 1.0 \pm 0.05$: $\Delta f = e \cdot 0.05 \approx 0.13591$.


In [3]:
x  = 1.0
dx = 0.05

delta_f = propagate_uncertainty(np.exp, x, dx)
exact   = np.exp(x) * dx

print(f"f(x) = exp(x)   at   x = {x} +/- {dx}")
print(f"  Propagated uncertainty  : {delta_f:.6f}")
print(f"  Exact                   : {exact:.6f}")
print(f"  Error                   : {abs(delta_f - exact):.2e}")


f(x) = exp(x)   at   x = 1.0 +/- 0.05
  Propagated uncertainty  : 0.135914
  Exact                   : 0.135914
  Error                   : 2.93e-12


---
## 2. `propagate_uncertainty` — Multivariate Case ($\mathbb{R}^n \to \mathbb{R}$)

For multi-variable functions, pass arrays for both `x` and `dx`. The two combination methods give different answers:
- `"statistical"` (default): assumes independent random errors — gives a smaller, more realistic estimate.
- `"absolute"`: assumes all errors act in the worst-case direction simultaneously — gives an upper bound.

### 2.1 Ohm's Law: $R = V / I$
We measure $V = 5.0 \pm 0.05$ V and $I = 2.0 \pm 0.02$ A.

Nominal: $R = 2.5\,\Omega$. 
Exact sensitivities: $\partial R/\partial V = 1/I = 0.5$,  $\partial R/\partial I = -V/I^2 = -1.25$.


In [4]:
f    = lambda v: v[0] / v[1]   # R = V / I
vals = np.array([5.0, 2.0])    # [V, I]
errs = np.array([0.05, 0.02])  # [dV, dI]

R         = f(vals)
delta_stat = propagate_uncertainty(f, vals, errs, method="statistical")
delta_abs  = propagate_uncertainty(f, vals, errs, method="absolute")

# Analytical comparison
dR_dV = 1 / vals[1]
dR_dI = -vals[0] / vals[1]**2
exact_stat = np.sqrt((dR_dV * errs[0])**2 + (dR_dI * errs[1])**2)
exact_abs  = abs(dR_dV * errs[0]) + abs(dR_dI * errs[1])

print(f"R = V / I   at   V = {vals[0]} +/- {errs[0]} V,  I = {vals[1]} +/- {errs[1]} A")
print(f"  Nominal R = {R:.4f} Ohm")
print()
print(f"  method='statistical'  :  dR = {delta_stat:.6f}  (exact = {exact_stat:.6f})")
print(f"  method='absolute'     :  dR = {delta_abs:.6f}  (exact = {exact_abs:.6f})")


R = V / I   at   V = 5.0 +/- 0.05 V,  I = 2.0 +/- 0.02 A
  Nominal R = 2.5000 Ohm

  method='statistical'  :  dR = 0.035355  (exact = 0.035355)
  method='absolute'     :  dR = 0.050000  (exact = 0.050000)


### 2.2 Kinetic Energy: $KE = \frac{1}{2} m v^2$
We measure $m = 2.0 \pm 0.01$ kg and $v = 3.0 \pm 0.05$ m/s.

Nominal: $KE = 9.0$ J.
Exact sensitivities: $\partial KE/\partial m = v^2/2 = 4.5$,  $\partial KE/\partial v = mv = 6.0$.


In [5]:
f    = lambda v: 0.5 * v[0] * v[1]**2   # KE = 1/2 * m * v^2
vals = np.array([2.0, 3.0])             # [m, v]
errs = np.array([0.01, 0.05])           # [dm, dv]

KE         = f(vals)
delta_stat = propagate_uncertainty(f, vals, errs, method="statistical")
delta_abs  = propagate_uncertainty(f, vals, errs, method="absolute")

# Individual error contributions
dKE_dm = 0.5 * vals[1]**2
dKE_dv = vals[0] * vals[1]
term_m = abs(dKE_dm * errs[0])
term_v = abs(dKE_dv * errs[1])

print(f"KE = 0.5*m*v^2   at   m = {vals[0]} +/- {errs[0]} kg,  v = {vals[1]} +/- {errs[1]} m/s")
print(f"  Nominal KE = {KE:.4f} J")
print()
print(f"  Error contributions:")
print(f"    From m : |dKE/dm * dm| = {term_m:.4f} J")
print(f"    From v : |dKE/dv * dv| = {term_v:.4f} J  (dominant!)")
print()
print(f"  dKE (statistical) = {delta_stat:.6f} J")
print(f"  dKE (absolute)    = {delta_abs:.6f} J")


KE = 0.5*m*v^2   at   m = 2.0 +/- 0.01 kg,  v = 3.0 +/- 0.05 m/s
  Nominal KE = 9.0000 J

  Error contributions:
    From m : |dKE/dm * dm| = 0.0450 J
    From v : |dKE/dv * dv| = 0.3000 J  (dominant!)

  dKE (statistical) = 0.303356 J
  dKE (absolute)    = 0.345000 J


---
## 3. `relative_uncertainty` — Fractional Error

`relative_uncertainty` computes $\delta f = |\Delta f / f(\mathbf{x})|$, the dimensionless fractional uncertainty. This is useful for comparing precision across different quantities and units.

$$\delta f = \frac{\Delta f}{|f(\mathbf{x})|}$$

### 3.1 Ohm's Law revisited — absolute vs. relative


In [6]:
f    = lambda v: v[0] / v[1]
vals = np.array([5.0, 2.0])
errs = np.array([0.05, 0.02])

delta_f = propagate_uncertainty(f, vals, errs)
rel_f   = relative_uncertainty(f, vals, errs)

print(f"R = V / I   at   V = {vals[0]} +/- {errs[0]} V,  I = {vals[1]} +/- {errs[1]} A")
print(f"  R              = {f(vals):.4f} Ohm")
print(f"  Absolute dR    = {delta_f:.6f} Ohm")
print(f"  Relative dR    = {rel_f:.4%}")


R = V / I   at   V = 5.0 +/- 0.05 V,  I = 2.0 +/- 0.02 A
  R              = 2.5000 Ohm
  Absolute dR    = 0.035355 Ohm
  Relative dR    = 1.4142%


### 3.2 Effect of Instrument Precision on Relative Uncertainty
How does the relative uncertainty of $R = V / I$ change as we improve the measurement precision by a factor of 10 each step?


In [7]:
f    = lambda v: v[0] / v[1]
vals = np.array([5.0, 2.0])

print(f"R = V / I   at   V = {vals[0]} V,  I = {vals[1]} A")
print(f"{'Precision factor':>18}  {'dV':>8}  {'dI':>8}  {'dR (abs)':>12}  {'dR (rel)':>10}")
print("-" * 64)
for exp in range(0, -5, -1):
    scale = 10**exp
    errs  = np.array([0.05, 0.02]) * scale
    d_abs = propagate_uncertainty(f, vals, errs)
    d_rel = relative_uncertainty(f, vals, errs)
    print(f"  {'x' + str(scale):>16}  {errs[0]:>8.5f}  {errs[1]:>8.5f}  {d_abs:>12.6f}  {d_rel:>10.4%}")


R = V / I   at   V = 5.0 V,  I = 2.0 A
  Precision factor        dV        dI      dR (abs)    dR (rel)
----------------------------------------------------------------
                x1   0.05000   0.02000      0.035355     1.4142%
              x0.1   0.00500   0.00200      0.003536     0.1414%
             x0.01   0.00050   0.00020      0.000354     0.0141%
            x0.001   0.00005   0.00002      0.000035     0.0014%
           x0.0001   0.00001   0.00000      0.000004     0.0001%


---
## 4. Edge Cases and Pitfalls

Understanding what can go wrong helps prevent silent errors in lab calculations.

### 4.1 Relative Uncertainty at $f(x) = 0$
Relative uncertainty is undefined when the nominal function value is zero.


In [8]:
try:
    relative_uncertainty(np.sin, 0.0, 0.01)  # sin(0) = 0 -> undefined
except ZeroDivisionError as e:
    print(f"ZeroDivisionError: {e}")
    print("  --> Use 'absolute' uncertainty when f(x) = 0.")


ZeroDivisionError: Nominal function value is zero; relative uncertainty is undefined.
  --> Use 'absolute' uncertainty when f(x) = 0.


### 4.2 Mismatched `x` and `dx` Shapes


In [9]:
try:
    propagate_uncertainty(
        lambda v: v[0] + v[1],
        x=np.array([1.0, 2.0]),
        dx=np.array([0.1, 0.2, 0.3]),  # wrong shape!
    )
except ValueError as e:
    print(f"ValueError: {e}")


ValueError: Value and uncertainty vectors must have the same dimension.


### 4.3 Non-Positive Step Size $h$


In [10]:
try:
    propagate_uncertainty(lambda x: x**2, 3.0, 0.1, h=0.0)
except ValueError as e:
    print(f"ValueError: {e}")


ValueError: Step size h must be strictly positive.


---
## 5. Putting It All Together: Pendulum Period Lab Scenario

A simple pendulum has period $T = 2\pi\sqrt{L/g}$, where:
- $L$ = pendulum length (m)
- $g$ = gravitational acceleration (m/s²)

We measure:
- $L = 1.000 \pm 0.005$ m (half-centimetre ruler precision)
- $g = 9.810 \pm 0.010$ m/s² (local gravity uncertainty)

We want to find $T$ and its propagated uncertainty $\Delta T$.


In [11]:
# Define the function
f    = lambda v: 2 * np.pi * np.sqrt(v[0] / v[1])  # T = 2*pi*sqrt(L/g)
vals = np.array([1.000, 9.810])                      # [L, g]
errs = np.array([0.005, 0.010])                      # [dL, dg]

T          = f(vals)
delta_T    = propagate_uncertainty(f, vals, errs, method="statistical")
rel_T      = relative_uncertainty(f, vals, errs, method="statistical")

print(f"Pendulum period  T = 2*pi*sqrt(L/g)")
print(f"  L = {vals[0]:.3f} +/- {errs[0]:.3f} m")
print(f"  g = {vals[1]:.3f} +/- {errs[1]:.3f} m/s^2")
print()
print(f"  T           = {T:.6f} s")
print(f"  Delta T     = {delta_T:.6f} s  (statistical)")
print(f"  Relative dT = {rel_T:.4%}")
print(f"  Result      : T = ({T:.4f} +/- {delta_T:.4f}) s")


Pendulum period  T = 2*pi*sqrt(L/g)
  L = 1.000 +/- 0.005 m
  g = 9.810 +/- 0.010 m/s^2

  T           = 2.006067 s
  Delta T     = 0.005118 s  (statistical)
  Relative dT = 0.2551%
  Result      : T = (2.0061 +/- 0.0051) s


### 5.1 Error Budget — Which Variable Dominates?

We can isolate each variable's contribution to the total uncertainty by setting all other uncertainties to zero.


In [12]:
names = ["L", "g"]

print(f"Error budget for T = 2*pi*sqrt(L/g)")
print(f"  Total Delta T = {delta_T:.6f} s")
print()
print(f"  {'Variable':>8}  {'Uncertainty':>12}  {'Contribution':>14}  {'% of total':>12}")
print("  " + "-" * 54)
for i, name in enumerate(names):
    isolated_errs = np.zeros_like(errs)
    isolated_errs[i] = errs[i]
    contribution = propagate_uncertainty(f, vals, isolated_errs, method="statistical")
    fraction = (contribution / delta_T) * 100
    print(f"  {name:>8}  +/- {errs[i]:.4f}      {contribution:>12.6f} s  {fraction:>10.1f}%")


Error budget for T = 2*pi*sqrt(L/g)
  Total Delta T = 0.005118 s

  Variable   Uncertainty    Contribution    % of total
  ------------------------------------------------------
         L  +/- 0.0050          0.005015 s        98.0%
         g  +/- 0.0100          0.001022 s        20.0%


### 5.2 Statistical vs. Absolute — Comparison


In [13]:
delta_stat = propagate_uncertainty(f, vals, errs, method="statistical")
delta_abs  = propagate_uncertainty(f, vals, errs, method="absolute")

print(f"Pendulum period uncertainty comparison:")
print(f"  T = {T:.6f} s")
print()
print(f"  Statistical (RSS)  : Delta T = {delta_stat:.6f} s   ({(delta_stat/T)*100:.4f}% relative)")
print(f"  Absolute (worst)   : Delta T = {delta_abs:.6f} s   ({(delta_abs/T)*100:.4f}% relative)")
print()
print(f"  The absolute method is {delta_abs/delta_stat:.2f}x more conservative than statistical.")


Pendulum period uncertainty comparison:
  T = 2.006067 s

  Statistical (RSS)  : Delta T = 0.005118 s   (0.2551% relative)
  Absolute (worst)   : Delta T = 0.006038 s   (0.3010% relative)

  The absolute method is 1.18x more conservative than statistical.
